# 07 - Evaluation & insights

In [6]:
import glob, json
import pandas as pd

rows = [json.load(open(f)) for f in sorted(glob.glob("../results/*.json"))]
df = pd.DataFrame(rows).set_index("name")

def col(c):
    """Colonna se esiste (solo alcune righe l'hanno), altrimenti tutta NaN."""
    return df[c] if c in df else pd.Series(float("nan"), index=df.index)

ref = df.loc["teacher_baseline"]

is_sparse = col("nonzero_params").notna()                   # righe del pruning non strutturato
params = col("nonzero_params").fillna(df["params"]).astype(int)   # non nulli per le righe sparse
macs_dense = col("macs_dense").fillna(df["macs"])           # df["macs"] è già sparso per le righe sparse

table = pd.DataFrame({
    "test acc (%)": (df["test_acc"] * 100).round(2),
    "Δ acc (pt)": ((df["test_acc"] - ref["test_acc"]) * 100).round(2),
    "params (non nulli)": params,
    "sparsity (%)": (col("sparsity") * 100).round(1),
    "size (MB)": df["size_mb"].round(3),
    "size x": (ref["size_mb"] / df["size_mb"]).round(1),
    "MACs (M)": (df["macs"] / 1e6).round(2),
    "MACs denso (M)": (macs_dense / 1e6).round(2),
    "Peak RAM (KB)": df["peak_act_kb"].round(2),
    "latency (ms)": df["latency_ms_median"].round(2),   # CPU, 1 thread
    "energy (µJ)": df["energy_uj"].round(2),
})
table.index = [n + " *" if s else n for n, s in zip(table.index, is_sparse)]

print("* pruning non strutturato: params = pesi non nulli; MACs ed energia sono sparsi \n"
      "(validi solo su HW che salta gli zeri); size = stima formato sparso; \n"
      "latency non misurata (con tensori densi non cambia).\n")
table.style.format(precision=2, na_rep="–")

* pruning non strutturato: params = pesi non nulli; MACs ed energia sono sparsi 
(validi solo su HW che salta gli zeri); size = stima formato sparso; 
latency non misurata (con tensori densi non cambia).



,test acc (%),Δ acc (pt),params (non nulli),sparsity (%),size (MB),size x,MACs (M),MACs denso (M),Peak RAM (KB),latency (ms),energy (µJ)
student_baseline,75.52,-13.76,110481,–,0.46,134.40,1.63,1.63,32.77,0.72,7.48
student_int8,75.46,-13.82,110021,–,0.11,547.50,1.63,1.63,8.19,–,0.49
student_struct,74.41,-14.87,93457,–,0.38,163.00,1.61,1.61,32.77,0.54,7.40
student_unstruct *,74.13,-15.15,66211,40.00,0.40,156.90,1.14,1.63,32.77,–,5.23
teacher_baseline,89.28,0.00,15530762,–,62.16,1.00,215.69,215.69,327.68,8.30,992.17
teacher_int8,89.20,-0.08,15525258,–,15.57,4.00,215.69,215.69,81.92,–,64.71
teacher_struct,87.94,-1.34,10146885,–,40.59,1.50,210.31,210.31,327.68,7.22,967.44
teacher_unstruct *,87.58,-1.70,6213603,60.00,37.27,1.70,110.69,215.69,327.68,–,509.18
